In [ ]:
import hashlib
import importlib
import json
import platform
import shutil
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

NOTEBOOK = "08_full_history_ingestion.ipynb"
notebook_started = time.perf_counter()

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (
        path for path in candidates
        if (path / "pyproject.toml").is_file()
        and (path / "configs/research_config.example.json").is_file()
        and (path / "src/eurusd_research/coverage_audit.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    entered = input("Full repository folder containing configs and src: ").strip().strip('"')
    if not entered:
        raise ValueError("Enter the full repository path.")
    REPO_ROOT = Path(entered).expanduser().resolve()

import numpy as np
import pandas as pd

from eurusd_research.config import load_config
from eurusd_research import features as feature_io
from eurusd_research.data import _sha256

config = load_config(REPO_ROOT)
RAW_ROOT = Path(config["raw_data_dir"]).resolve()
feature_settings = config["sampling_features"]

audit_collection_path = (
    REPO_ROOT / config["active_coverage_audit_manifest"]
).resolve()
audit_collection = json.loads(
    audit_collection_path.read_text(encoding="utf-8")
)
if audit_collection["status"] != "complete":
    raise RuntimeError("A completed Notebook 07 audit is required.")

audit_references = {
    item["audit_job_id"]: item
    for item in audit_collection["identity"]["source_audits"]
}
audit_states = []
for relative in audit_collection["source_manifests"]:
    path = REPO_ROOT / relative
    state = json.loads(path.read_text(encoding="utf-8"))
    reference = audit_references[state["job_id"]]
    if _sha256(path) != reference["audit_manifest_sha256"]:
        raise RuntimeError("A source audit manifest changed.")
    if state["status"] != "complete" or state["utc_min"] is None:
        raise RuntimeError("A source audit is incomplete or lacks usable timestamps.")
    audit_states.append(state)

audit_states.sort(key=lambda state: pd.Timestamp(state["utc_min"]))

parameters = {
    key: audit_collection["identity"]["audit_parameters"][key]
    for key in (
        "timezone", "timestamp_format", "pip_size",
        "gap_seconds", "extreme_spread_pips", "block_bytes",
    )
}

history_settings = config.get("history_ingestion", {
    "minimum_free_gib": 16,
    "quote_storage_multiplier": 2.0,
    "scratch_reserve_gib": 8,
    "final_test_complete_months": 12,
})

versions = {
    name: version(name)
    for name in ("numpy", "pandas", "pyarrow", "duckdb", "pytest", "tzdata", "pytz")
}
new_sources = {}

print("NOTEBOOK_08_PREFLIGHT")
print("Repository:", REPO_ROOT)
print("Raw-data folder:", RAW_ROOT)
print("Audited exports:", len(audit_states))
print("Expected records:", sum(state["rows_scanned"] for state in audit_states))
print("Source timezone:", parameters["timezone"])
print("No installation or restart is required.")

In [ ]:
new_sources["src/eurusd_research/history.py"] = r'''
"""Audited full-source ingestion and portable canonical quote collections."""

import hashlib
import json
import os
import tempfile
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from . import features as io
from .data import _sha256, _writer_lock, ingest_quotes


def _utc(value):
    value = pd.Timestamp(value)
    if pd.isna(value) or value.tzinfo is None:
        raise ValueError("A valid timezone-aware timestamp is required.")
    return value.tz_convert("UTC")


def _relative(root, path):
    path = Path(path).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError("An artifact path escapes the repository.")
    return path.relative_to(root).as_posix()


def validate_audits(audits):
    if not audits:
        raise ValueError("At least one completed source audit is required.")
    ordered = sorted(audits, key=lambda item: _utc(item["utc_min"]))
    if len({item["source_sha256"] for item in ordered}) != len(ordered):
        raise ValueError("Duplicate complete source content requires review.")

    previous_end = None
    for item in ordered:
        if (
            item["status"] != "complete"
            or item["rows_scanned"] <= 0
            or item["valid_timestamp_rows"] != item["rows_scanned"]
            or item["unassigned_timestamp_rows"] != 0
            or item["source_order_backsteps"] != 0
            or item["repeated_valid_timestamp_steps"] != 0
            or item["adjacent_exact_export_duplicate_rows"] != 0
        ):
            raise ValueError("Source timestamps or ordering require review.")
        start, end = _utc(item["utc_min"]), _utc(item["utc_max"])
        if (
            start > end
            or item["first_record"]["source_row"] != 1
            or item["last_record"]["source_row"] != item["rows_scanned"]
            or _utc(item["first_record"]["timestamp_utc"]) != start
            or _utc(item["last_record"]["timestamp_utc"]) != end
        ):
            raise ValueError("Audited source endpoints are inconsistent.")
        if previous_end is not None and start <= previous_end:
            raise ValueError("Source ranges overlap or share a timestamp; review required.")
        previous_end = end
    return ordered


def ingest_audited_source(raw_root, repository_root, audit, parameters):
    root = Path(repository_root).resolve()
    raw_root = Path(raw_root).resolve()
    source = (raw_root / audit["source_name"]).resolve()
    if not source.is_relative_to(raw_root) or not source.is_file():
        raise FileNotFoundError(source)
    if (
        source.stat().st_size != audit["identity"]["source_bytes"]
        or _sha256(source) != audit["source_sha256"]
    ):
        raise RuntimeError("Source content differs from the audited snapshot.")

    start = _utc(audit["utc_min"]).normalize()
    end = _utc(audit["utc_max"]).normalize() + pd.Timedelta(days=1)
    state = ingest_quotes(
        source,
        root / "data/interim/ingestion",
        root / "artifacts/manifests/ingestion",
        start, end,
        **parameters,
    )
    if (
        state["identity"]["source_sha256"] != audit["source_sha256"]
        or state["rows_scanned"] != audit["rows_scanned"]
        or state["rows_in_scope"] != audit["valid_timestamp_rows"]
        or state["rows_unassigned"] != 0
        or state["rows_outside_scope"] != 0
        or _utc(state["scope_utc_min"]) != _utc(audit["utc_min"])
        or _utc(state["scope_utc_max"]) != _utc(audit["utc_max"])
        or state["source_flags"] != audit["flags"]
    ):
        raise RuntimeError("Full-source ingestion differs from its audited results.")
    return state


def _bounds(parquet):
    field = parquet.schema_arrow.field("timestamp_utc")
    if not pa.types.is_timestamp(field.type) or field.type.tz != "UTC":
        raise RuntimeError("Canonical quote timestamps must explicitly carry UTC.")
    column = parquet.schema_arrow.get_field_index("timestamp_utc")
    minima, maxima = [], []
    for number in range(parquet.metadata.num_row_groups):
        stats = parquet.metadata.row_group(number).column(column).statistics
        if stats is None or not stats.has_min_max:
            result = pc.min_max(
                parquet.read(columns=["timestamp_utc"])["timestamp_utc"]
            ).as_py()
            minima, maxima = [result["min"]], [result["max"]]
            break
        minima.append(stats.min)
        maxima.append(stats.max)

    def convert(value):
        value = pd.Timestamp(value)
        # The field's verified Arrow type establishes UTC for naive statistics.
        return value.tz_localize("UTC") if value.tzinfo is None else value.tz_convert("UTC")

    return (
        min(convert(value) for value in minima).isoformat(),
        max(convert(value) for value in maxima).isoformat(),
    )


def _boundary_overlay(original_path, target, first_timestamp, previous_last, gap_limit):
    original_path, target = Path(original_path).resolve(), Path(target).resolve()
    if original_path == target:
        raise RuntimeError("A boundary overlay cannot replace an original partition.")
    original_hash = _sha256(original_path)
    original = pq.ParquetFile(original_path).read()
    timestamp_type = original.schema.field("timestamp_utc").type
    mask = pc.equal(
        original["timestamp_utc"],
        pa.scalar(_utc(first_timestamp).to_pydatetime(), type=timestamp_type),
    )
    if pc.sum(pc.cast(mask, pa.int64())).as_py() != 1:
        raise RuntimeError("The first source quote is not unique in its first partition.")
    old_gap = pc.filter(original["source_gap_seconds"], mask)[0].as_py()
    if old_gap is not None and not pd.isna(old_gap):
        raise RuntimeError("The native first-source gap was unexpectedly populated.")

    gap = float((_utc(first_timestamp) - _utc(previous_last)).total_seconds())
    if gap <= 0:
        raise RuntimeError("A source boundary is not strictly chronological.")

    patched = original
    replacements = {
        "source_gap_seconds": gap,
        "flag_source_gap": gap > gap_limit,
    }
    for name, value in replacements.items():
        field = original.schema.field(name)
        replacement = pc.if_else(
            mask, pa.scalar(value, type=field.type), original[name]
        )
        patched = patched.set_column(
            patched.schema.get_field_index(name), field, replacement
        )

    for name in original.column_names:
        if name not in replacements:
            if not original[name].equals(patched[name]):
                raise RuntimeError("A boundary overlay altered an unrelated field.")
        elif not pc.filter(original[name], pc.invert(mask)).equals(
            pc.filter(patched[name], pc.invert(mask))
        ):
            raise RuntimeError("A boundary overlay altered another observation.")

    if io._schema_hash(original.schema) != io._schema_hash(patched.schema):
        raise RuntimeError("Boundary overlay schema changed.")

    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="wb", prefix=".boundary-", suffix=".tmp",
            dir=target.parent, delete=False,
        ) as handle:
            temporary = Path(handle.name)
            pq.write_table(patched, handle, compression="zstd")
            handle.flush()
            os.fsync(handle.fileno())
        io._replace_path(temporary, target)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

    if _sha256(original_path) != original_hash:
        raise RuntimeError("An original source partition changed.")
    return {
        "rows": patched.num_rows,
        "bytes": target.stat().st_size,
        "sha256": _sha256(target),
        "schema_sha256": io._schema_hash(patched.schema),
    }
'''.lstrip()

print("Prepared audited source ingestion and boundary overlays.")

new_sources["src/eurusd_research/history.py"] += r'''


def verify_collection(repository_root, collection, verify_hashes=True):
    root = Path(repository_root).resolve()
    total = 0
    common_schema = None
    previous_end = None
    required = set(io.QUOTE_COLUMNS + ["flag_source_gap"])

    for member in collection["members"]:
        path = (root / member["ingestion_manifest"]).resolve()
        if (
            not path.is_relative_to(root) or not path.is_file()
            or _sha256(path) != member["ingestion_manifest_sha256"]
        ):
            raise RuntimeError("A source ingestion manifest changed.")

    for part in collection["parts"]:
        path = (root / part["path"]).resolve()
        if (
            part["kind"] != "quotes"
            or not path.is_relative_to(root) or not path.is_file()
            or path.stat().st_size != part["bytes"]
        ):
            raise RuntimeError("Canonical partition location or size failed.")
        if verify_hashes and _sha256(path) != part["sha256"]:
            raise RuntimeError("Canonical partition checksum failed.")
        parquet = pq.ParquetFile(path)
        schema = parquet.schema_arrow
        schema_hash = io._schema_hash(schema)
        if not required.issubset(schema.names):
            raise RuntimeError("A canonical partition lacks required fields.")
        if common_schema is None:
            common_schema = schema_hash
        if schema_hash != common_schema or parquet.metadata.num_rows != part["rows"]:
            raise RuntimeError("Canonical schemas or row counts differ.")
        if _bounds(parquet) != (part["min_utc"], part["max_utc"]):
            raise RuntimeError("Canonical partition timestamp bounds differ.")
        if previous_end is not None and _utc(part["min_utc"]) <= previous_end:
            raise RuntimeError("Canonical partition time ranges are not disjoint.")
        previous_end = _utc(part["max_utc"])
        total += part["rows"]

    if total != collection["rows_in_scope"] or total != collection["rows_scanned"]:
        raise RuntimeError("Canonical collection row conservation failed.")
    if not collection["parts"] or collection["rows_unassigned"] != 0:
        raise RuntimeError("The collection is empty or has unresolved timestamps.")
    if (
        _utc(collection["parts"][0]["min_utc"]) != _utc(collection["scope_utc_min"])
        or _utc(collection["parts"][-1]["max_utc"]) != _utc(collection["scope_utc_max"])
    ):
        raise RuntimeError("Canonical collection coverage differs.")


def build_collection(repository_root, audits, source_states, parameters,
                     audit_collection_path):
    root = Path(repository_root).resolve()
    audits = validate_audits(audits)
    by_hash = {
        state["identity"]["source_sha256"]: state for state in source_states
    }
    if len(by_hash) != len(audits) or set(by_hash) != {
        audit["source_sha256"] for audit in audits
    }:
        raise RuntimeError("Expected one complete ingestion per audited source.")

    members = []
    for rank, audit in enumerate(audits):
        state = by_hash[audit["source_sha256"]]
        if (
            state["status"] != "complete"
            or state["rows_in_scope"] != audit["rows_scanned"]
            or state["rows_unassigned"] != 0
            or state["rows_outside_scope"] != 0
            or state["source_flags"] != audit["flags"]
        ):
            raise RuntimeError("A source generation does not match its audit.")
        previous = None if rank == 0 else audits[rank - 1]["utc_max"]
        gap = None if previous is None else float(
            (_utc(audit["utc_min"]) - _utc(previous)).total_seconds()
        )
        members.append({
            "source_rank": rank,
            "source_file": audit["source_name"],
            "source_sha256": audit["source_sha256"],
            "rows": audit["rows_scanned"],
            "utc_min": audit["utc_min"], "utc_max": audit["utc_max"],
            "previous_source_last_utc": previous,
            "boundary_gap_seconds": gap,
            "ingestion_manifest": _relative(root, state["manifest_path"]),
            "ingestion_manifest_sha256": _sha256(state["manifest_path"]),
        })

    identity = {
        "audit_collection": _relative(root, audit_collection_path),
        "audit_collection_sha256": _sha256(audit_collection_path),
        "sources": members,
        "parameters": parameters,
        "start_utc": _utc(audits[0]["utc_min"]).normalize().isoformat(),
        "end_utc": (
            _utc(audits[-1]["utc_max"]).normalize() + pd.Timedelta(days=1)
        ).isoformat(),
        "boundary_policy": "first later-source quote uses preceding source endpoint",
        "implementation_sha256": hashlib.sha256(b"".join(
            Path(__file__).with_name(name).read_bytes()
            for name in ("history.py", "data.py", "quality.py", "features.py")
        )).hexdigest(),
        "versions": {
            name: version(name) for name in ("pandas", "pyarrow", "duckdb")
        },
    }
    job_id = hashlib.sha256(
        json.dumps(identity, sort_keys=True).encode()
    ).hexdigest()[:24]
    manifest_root = root / "artifacts/manifests/history"
    manifest = manifest_root / f"collection-{job_id}.json"

    with _writer_lock(manifest_root):
        if manifest.exists():
            saved = json.loads(manifest.read_text(encoding="utf-8"))
            if saved["identity"] != identity:
                raise RuntimeError("Existing canonical collection identity differs.")
            verify_collection(root, saved)
            return {**saved, "cache_hit": True}

        parts, patches, flags = [], [], {}
        for member, audit in zip(members, audits):
            state = by_hash[member["source_sha256"]]
            source_parts = [
                part for part in state["parts"] if part["kind"] == "quotes"
            ]
            for name, count in state["source_flags"].items():
                flags[name] = flags.get(name, 0) + int(count)

            for number, native in enumerate(source_parts):
                original_path = Path(state["output_dir"]) / native["path"]
                selected_path = original_path
                record = dict(native)

                if number == 0 and member["source_rank"] > 0:
                    first = _utc(member["utc_min"])
                    selected_path = (
                        root / "data/interim/history" / job_id
                        / "boundary_overlays"
                        / f"year={first.year:04d}/month={first.month:02d}"
                        / f"source-{member['source_rank']:03d}.parquet"
                    )
                    metadata = _boundary_overlay(
                        original_path, selected_path, member["utc_min"],
                        member["previous_source_last_utc"],
                        parameters["gap_seconds"],
                    )
                    if metadata["rows"] != native["rows"]:
                        raise RuntimeError("Boundary replacement changed row count.")
                    record.update(metadata)
                    patches.append({
                        "source_file": member["source_file"],
                        "source_sha256": member["source_sha256"],
                        "first_timestamp_utc": member["utc_min"],
                        "previous_source_last_utc":
                            member["previous_source_last_utc"],
                        "gap_seconds": member["boundary_gap_seconds"],
                        "flag_source_gap": (
                            member["boundary_gap_seconds"] > parameters["gap_seconds"]
                        ),
                        "original_partition": _relative(root, original_path),
                        "original_partition_sha256": native["sha256"],
                        "overlay_partition": _relative(root, selected_path),
                    })
                    flags["flag_source_gap"] += int(
                        member["boundary_gap_seconds"] > parameters["gap_seconds"]
                    )

                parquet = pq.ParquetFile(selected_path)
                minimum, maximum = _bounds(parquet)
                record.update({
                    "path": _relative(root, selected_path),
                    "kind": "quotes",
                    "source_rank": member["source_rank"],
                    "min_utc": minimum, "max_utc": maximum,
                })
                parts.append(record)

        parts.sort(key=lambda part: _utc(part["min_utc"]))
        total = sum(audit["rows_scanned"] for audit in audits)
        collection = {
            "kind": "history_collection", "status": "complete",
            "job_id": job_id, "identity": identity,
            "members": members, "parts": parts,
            "boundary_patches": patches,
            "rows_scanned": total, "rows_in_scope": total,
            "rows_unassigned": 0, "rows_outside_scope": 0,
            "source_utc_min": audits[0]["utc_min"],
            "source_utc_max": audits[-1]["utc_max"],
            "scope_utc_min": audits[0]["utc_min"],
            "scope_utc_max": audits[-1]["utc_max"],
            "source_flags": flags, "scope_flags": dict(flags),
            "manifest_path": _relative(root, manifest),
        }
        verify_collection(root, collection)
        io.write_json(manifest, collection)
        return {**collection, "cache_hit": False}


def load_collection(repository_root, manifest_path, verify=True):
    root = Path(repository_root).resolve()
    path = Path(manifest_path)
    path = path.resolve() if path.is_absolute() else (root / path).resolve()
    collection = json.loads(path.read_text(encoding="utf-8"))
    if collection["kind"] != "history_collection" or collection["status"] != "complete":
        raise ValueError("A completed history collection is required.")
    if verify:
        verify_collection(root, collection)
    return {
        **collection,
        "output_dir": str(root),
        "manifest_path": str(path),
    }


def quote_files(repository_root, collection, start=None, end=None,
                include_predecessor=True):
    root = Path(repository_root).resolve()
    start = None if start is None else _utc(start)
    end = None if end is None else _utc(end)
    if start is not None and end is not None and start >= end:
        raise ValueError("Invalid quote-file selection range.")

    selected, predecessor = [], None
    for part in collection["parts"]:
        if start is not None and _utc(part["max_utc"]) < start:
            predecessor = part
            continue
        if end is not None and _utc(part["min_utc"]) >= end:
            continue
        selected.append(part)

    if start is not None and include_predecessor and predecessor is not None:
        selected.insert(0, predecessor)
    return [str(root / part["path"]) for part in selected]


def create_local_binding(repository_root, manifest_path, verify=True):
    root = Path(repository_root).resolve()
    collection = load_collection(root, manifest_path, verify=verify)
    collection_path = Path(collection["manifest_path"])
    collection_hash = _sha256(collection_path)
    binding_id = hashlib.sha256(
        f"{collection_hash}|{root}".encode()
    ).hexdigest()[:24]
    path = root / "artifacts/manifests/history_bindings" / f"{binding_id}.json"

    payload = {
        **collection,
        "kind": "history_collection_binding",
        "manifest_path": str(path),
        "canonical_collection_manifest": _relative(root, collection_path),
        "canonical_collection_sha256": collection_hash,
        "identity": {
            **collection["identity"],
            "canonical_collection_sha256": collection_hash,
        },
    }
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != payload:
            raise RuntimeError("Existing local history binding differs.")
    else:
        io.write_json(path, payload)
    return path
'''

print("Prepared canonical collection, indexed selection, and local binding.")

new_sources["tests/test_history.py"] = r'''
import copy
import json

import pandas as pd
import pytest

from eurusd_research import features as io
from eurusd_research.coverage_audit import audit_source
from eurusd_research.data import _sha256, _verify_parts, ingest_quotes
from eurusd_research.history import (
    build_collection, create_local_binding, ingest_audited_source,
    load_collection, quote_files, validate_audits, verify_collection,
)


@pytest.fixture
def built(tmp_path):
    root = tmp_path / "project"
    raw_root = root / "data/raw"
    raw_root.mkdir(parents=True)

    columns = ["Time (EET)", "Ask", "Bid", "AskVolume", "BidVolume"]
    first = pd.DataFrame([
        ["2020.01.31 23:59:59.000", "1.1002", "1.1000", "1", "2"],
        ["2020.02.01 00:00:01.000", "1.1003", "1.1001", "1", "2"],
    ], columns=columns)
    second = pd.DataFrame([
        ["2020.02.01 00:00:45.000", "1.1000", "1.1000", "0", "0"],
        ["2020.02.01 00:00:46.000", "1.0999", "1.1000", "1", "2"],
        ["2020.02.01 00:00:47.000", "1.1030", "1.1000", "1", "2"],
    ], columns=columns)
    first.to_csv(raw_root / "a.csv", index=False)
    second.to_csv(raw_root / "b.csv", index=False)
    params = {
        "timezone": "UTC", "timestamp_format": "%Y.%m.%d %H:%M:%S.%f",
        "pip_size": 0.0001, "gap_seconds": 30,
        "extreme_spread_pips": 20, "block_bytes": 1024,
    }
    audit_params = {
        **params, "checkpoint_batches": 1, "progress_batches": 100,
        "max_gap_examples": 5, "max_unassigned_examples": 5,
        "maximum_dataframe_mib": 20,
    }
    audits = [
        audit_source(raw_root / name, root, audit_params)
        for name in ("a.csv", "b.csv")
    ]
    states = [
        ingest_audited_source(raw_root, root, audit, params)
        for audit in audits
    ]
    audit_reference = root / "artifacts/test-audit.json"
    io.write_json(audit_reference, {"source_audits": [item["job_id"] for item in audits]})
    collection = build_collection(root, audits, states, params, audit_reference)
    return root, raw_root, audits, states, params, collection, first, second


def read_collection(root, collection):
    connection = io._connect(
        quote_files(root, collection), "quotes", root / "working",
        {"duckdb_memory_limit": "128MB", "duckdb_temp_limit": "1GB"},
    )
    try:
        return connection.execute(
            "SELECT * FROM quotes ORDER BY timestamp_utc, source_row"
        ).fetchdf()
    finally:
        connection.close()


def test_overlap_requires_review(built):
    _, _, audits, _, _, _, _, _ = built
    changed = copy.deepcopy(audits)
    changed[1]["utc_min"] = changed[0]["utc_max"]
    changed[1]["first_record"]["timestamp_utc"] = changed[0]["utc_max"]
    with pytest.raises(ValueError, match="overlap"):
        validate_audits(changed)


def test_unassigned_timestamp_requires_review(built):
    _, _, audits, _, _, _, _, _ = built
    changed = copy.deepcopy(audits)
    changed[0]["unassigned_timestamp_rows"] = 1
    with pytest.raises(ValueError):
        validate_audits(changed)


def test_boundary_gap_and_problematic_quotes_are_preserved(built):
    root, _, _, _, _, collection, _, _ = built
    frame = read_collection(root, collection)
    assert len(frame) == 5
    assert frame.loc[2, "source_gap_seconds"] == 44.0
    assert frame.loc[2, "flag_source_gap"]
    assert frame["flag_locked"].sum() == 1
    assert frame["flag_zero_total_size"].sum() == 1
    assert frame["flag_crossed"].sum() == 1
    assert frame["flag_extreme_spread"].sum() == 1


def test_split_and_combined_source_normalization_agree(built):
    root, raw_root, _, _, params, collection, first, second = built
    combined_path = raw_root / "combined.csv"
    pd.concat([first, second], ignore_index=True).to_csv(combined_path, index=False)
    combined = ingest_quotes(
        combined_path, root / "combined-data", root / "combined-manifests",
        "2020-01-31T00:00:00Z", "2020-02-02T00:00:00Z", **params,
    )
    files = [
        str(pd_path)
        for part in combined["parts"]
        if part["kind"] == "quotes"
        for pd_path in [__import__("pathlib").Path(combined["output_dir"]) / part["path"]]
    ]
    connection = io._connect(
        files, "quotes", root / "combined-working",
        {"duckdb_memory_limit": "128MB", "duckdb_temp_limit": "1GB"},
    )
    try:
        expected = connection.execute(
            "SELECT * FROM quotes ORDER BY timestamp_utc, source_row"
        ).fetchdf()
    finally:
        connection.close()
    actual = read_collection(root, collection)
    excluded = {"source_file", "source_sha256", "source_row"}
    columns = [name for name in expected if name not in excluded]
    pd.testing.assert_frame_equal(
        expected[columns], actual[columns],
        check_dtype=False, rtol=1e-10, atol=1e-12,
    )


def test_native_partition_is_unchanged(built):
    root, _, _, _, _, collection, _, _ = built
    patch = collection["boundary_patches"][0]
    original = root / patch["original_partition"]
    assert _sha256(original) == patch["original_partition_sha256"]


def test_binding_preserves_portable_manifest_and_legacy_integrity(built):
    root, _, _, _, _, collection, _, _ = built
    path = root / collection["manifest_path"]
    before = _sha256(path)
    binding_path = create_local_binding(root, path)
    binding = json.loads(binding_path.read_text(encoding="utf-8"))
    _verify_parts(binding)
    assert _sha256(path) == before
    assert binding["rows_in_scope"] == 5
    assert load_collection(root, path)["output_dir"] == str(root.resolve())


def test_changed_collection_partition_is_detected(built):
    root, _, _, _, _, collection, _, _ = built
    path = root / collection["parts"][0]["path"]
    path.write_bytes(b"changed")
    with pytest.raises(RuntimeError):
        verify_collection(root, collection)
'''.lstrip()

# Keep the fixture's file-list construction straightforward.
new_sources["tests/test_history.py"] = (
    new_sources["tests/test_history.py"]
    .replace("import json\n", "import json\nfrom pathlib import Path\n", 1)
    .replace(
        '    files = [\n'
        '        str(pd_path)\n'
        '        for part in combined["parts"]\n'
        '        if part["kind"] == "quotes"\n'
        '        for pd_path in [__import__("pathlib").Path(combined["output_dir"]) / part["path"]]\n'
        '    ]\n',
        '    files = [\n'
        '        str(Path(combined["output_dir"]) / part["path"])\n'
        '        for part in combined["parts"] if part["kind"] == "quotes"\n'
        '    ]\n',
    )
)

print("Prepared focused full-history ingestion tests.")



In [ ]:
for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if path.exists():
        existing = path.read_text(encoding="utf-8").replace("\r\n", "\n")
        if existing != source:
            raise RuntimeError(f"Existing source differs: {relative}. Stop.")

for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(source)
        print("CREATED", relative)
    else:
        print("REUSED", relative)

importlib.invalidate_caches()
probe = subprocess.run(
    [
        sys.executable, "-c",
        "from pathlib import Path; from eurusd_research import history; "
        "print(Path(history.__file__).resolve())",
    ],
    cwd=REPO_ROOT / "notebooks",
    capture_output=True, text=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("History-module import failed.")
if Path(probe.stdout.strip()).resolve() != (
    REPO_ROOT / "src/eurusd_research/history.py"
).resolve():
    raise RuntimeError("Imported a different history module.")

test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--color=no"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode:
    raise RuntimeError("Tests failed. Stop and return Cell 10 output.")
test_summary = test_run.stdout.strip().splitlines()[-1]

from eurusd_research.history import (
    build_collection, create_local_binding, ingest_audited_source,
    load_collection, quote_files, validate_audits,
)

for relative in (
    "data/interim/history/example.parquet",
    "artifacts/manifests/history/example.json",
    "artifacts/manifests/history_bindings/example.json",
    "reports/private/notebook08_execution.json",
    "reports/tables/history/example.csv",
):
    checked = subprocess.run(
        ["git", "check-ignore", "--no-index", "-q", "--", relative],
        cwd=REPO_ROOT, capture_output=True, text=True,
    )
    if checked.returncode != 0:
        raise RuntimeError(f"Generated path is not ignored: {relative}")

audit_states = validate_audits(audit_states)
print("PASS: tests, imports, ignore rules, and audited source assumptions.")

In [ ]:
expected_names = {state["source_name"] for state in audit_states}
actual_names = {
    path.name for path in RAW_ROOT.glob("EURUSD_Ticks_*.csv") if path.is_file()
}
if actual_names != expected_names:
    raise RuntimeError("Raw inventory differs from the audit. Review or rerun the audit.")

scope_table = pd.DataFrame([
    {
        "source_file": state["source_name"],
        "rows": state["rows_scanned"],
        "first_utc": state["utc_min"],
        "last_utc": state["utc_max"],
    }
    for state in audit_states
])

storage_records = [
    item for item in audit_collection["outputs"]
    if Path(item["path"]).name == "storage_projection.csv"
]
if len(storage_records) != 1:
    raise RuntimeError("The audited storage projection is missing.")
storage_path = REPO_ROOT / storage_records[0]["path"]
if _sha256(storage_path) != storage_records[0]["sha256"]:
    raise RuntimeError("The audited storage projection changed.")
storage_projection = pd.read_csv(storage_path)
quote_projection = float(storage_projection.loc[
    storage_projection["output"].eq("canonical_quotes"), "projected_bytes"
].iloc[0])

required_free_bytes = max(
    history_settings["minimum_free_gib"] * 1024**3,
    quote_projection * history_settings["quote_storage_multiplier"]
    + history_settings["scratch_reserve_gib"] * 1024**3,
)
free_bytes = shutil.disk_usage(REPO_ROOT).free
if free_bytes < required_free_bytes:
    raise RuntimeError("Available disk space is below the ingestion planning budget.")

print("FULL_SOURCE_INGESTION_SCOPE")
print(scope_table.to_string(index=False))
print("Expected canonical records:", int(scope_table["rows"].sum()))
print("Planning free-space requirement GiB:", round(required_free_bytes / 1024**3, 2))
print("Current free disk GiB:", round(free_bytes / 1024**3, 2))

In [ ]:
from eurusd_research.coverage_audit import audit_source

demo_root = REPO_ROOT / "data/interim/history_demo"
demo_raw = demo_root / "inputs"
demo_raw.mkdir(parents=True, exist_ok=True)

columns = ["Time (EET)", "Ask", "Bid", "AskVolume", "BidVolume"]
demo_frames = {
    "a.csv": pd.DataFrame([
        ["2020.01.31 23:59:59.000", "1.1002", "1.1000", "1", "2"],
        ["2020.02.01 00:00:01.000", "1.1003", "1.1001", "1", "2"],
    ], columns=columns),
    "b.csv": pd.DataFrame([
        ["2020.02.01 00:00:45.000", "1.1000", "1.1000", "0", "0"],
        ["2020.02.01 00:00:46.000", "1.0999", "1.1000", "1", "2"],
        ["2020.02.01 00:00:47.000", "1.1030", "1.1000", "1", "2"],
    ], columns=columns),
}
for name, frame in demo_frames.items():
    path = demo_raw / name
    text = frame.to_csv(index=False, lineterminator="\n")
    if path.exists():
        if path.read_text(encoding="utf-8") != text:
            raise RuntimeError("Existing synthetic input differs.")
    else:
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(text)

demo_parameters = {**parameters, "timezone": "UTC", "block_bytes": 1024}
demo_audit_parameters = {
    **demo_parameters,
    "checkpoint_batches": 1, "progress_batches": 100,
    "max_gap_examples": 5, "max_unassigned_examples": 5,
    "maximum_dataframe_mib": 20,
}
demo_audits = [
    audit_source(demo_raw / name, REPO_ROOT, demo_audit_parameters)
    for name in demo_frames
]
demo_sources = [
    ingest_audited_source(demo_raw, REPO_ROOT, audit, demo_parameters)
    for audit in demo_audits
]
demo_reference = demo_root / "audit_reference.json"
feature_io.write_json(
    demo_reference,
    {"source_audit_jobs": [state["job_id"] for state in demo_audits]},
)
demo_collection = build_collection(
    REPO_ROOT, demo_audits, demo_sources, demo_parameters, demo_reference
)

connection = feature_io._connect(
    quote_files(REPO_ROOT, demo_collection),
    "quotes", demo_root / "query", feature_settings,
)
try:
    demo_excerpt = connection.execute("""
        SELECT timestamp_utc, source_file, source_row,
               source_gap_seconds, flag_source_gap,
               flag_crossed, flag_zero_total_size, flag_extreme_spread
        FROM quotes ORDER BY timestamp_utc, source_row
    """).fetchdf()
finally:
    connection.close()

assert len(demo_excerpt) == 5
assert demo_excerpt.iloc[2]["source_gap_seconds"] == 44.0
assert demo_excerpt.iloc[2]["flag_source_gap"]

print("SYNTHETIC_CANONICAL_COLLECTION")
print(demo_excerpt.to_string(index=False))
print("PASS: boundary gap corrected; all five observations retained.")

In [ ]:
RUN_FULL_HISTORY_INGESTION = True

if not RUN_FULL_HISTORY_INGESTION:
    raise RuntimeError("Full-history ingestion disabled; later cells require its outputs.")

ingestion_started = time.perf_counter()
source_states = []
source_execution_records = []

for number, audit in enumerate(audit_states, start=1):
    print(f"\nSOURCE {number}/{len(audit_states)}: {audit['source_name']}")
    began = time.perf_counter()
    state = ingest_audited_source(
        RAW_ROOT, REPO_ROOT, audit, parameters
    )
    source_states.append(state)
    source_execution_records.append({
        "source_file": audit["source_name"],
        "ingestion_job_id": state["job_id"],
        "rows": state["rows_in_scope"],
        "parquet_bytes": sum(part["bytes"] for part in state["parts"]),
        "runtime_seconds_this_execution": round(time.perf_counter() - began, 3),
        "cache_hit": state["cache_hit"],
    })
    print("PASS:", state["rows_in_scope"], "records retained.")

history_collection = build_collection(
    REPO_ROOT,
    audit_states,
    source_states,
    parameters,
    audit_collection_path,
)
history_manifest_path = REPO_ROOT / history_collection["manifest_path"]
ingestion_runtime = time.perf_counter() - ingestion_started

print("\nCANONICAL_COLLECTION_CREATED")
print("Collection:", history_collection["job_id"])
print("Canonical records:", history_collection["rows_in_scope"])
print("Indexed partitions:", len(history_collection["parts"]))
print("Boundary overlays:", len(history_collection["boundary_patches"]))
print("Runtime this execution:", round(ingestion_runtime, 3), "seconds")

In [ ]:
canonical = load_collection(
    REPO_ROOT, history_manifest_path, verify=False
)
connection = feature_io._connect(
    quote_files(REPO_ROOT, canonical),
    "quotes",
    REPO_ROOT / "artifacts/manifests/history/diagnostics",
    feature_settings,
)

price_condition = (
    "isfinite(ask) AND isfinite(bid) AND ask > 0 AND bid > 0 AND ask >= bid"
)
size_condition = (
    f"({price_condition}) AND isfinite(ask_size) AND isfinite(bid_size) "
    "AND ask_size >= 0 AND bid_size >= 0 "
    "AND isfinite(ask_size + bid_size) AND ask_size + bid_size > 0"
)

try:
    rows, first_time, last_time, sources, missing_gaps = connection.execute("""
        SELECT count(*), min(timestamp_utc), max(timestamp_utc),
               count(DISTINCT source_sha256),
               count(*) FILTER (WHERE source_gap_seconds IS NULL)
        FROM quotes
    """).fetchone()

    flag_names = sorted(canonical["source_flags"])
    flag_row = connection.execute(
        "SELECT " + ", ".join(
            f"count(*) FILTER (WHERE {name})" for name in flag_names
        ) + " FROM quotes"
    ).fetchone()
    canonical_flags = {
        name: int(count) for name, count in zip(flag_names, flag_row)
    }

    canonical_years = connection.execute(f"""
        SELECT extract(year FROM timestamp_utc)::INTEGER AS year_utc,
               count(*) AS quote_rows,
               count(*) FILTER (WHERE {price_condition})
                   AS price_eligible_quote_rows,
               count(*) FILTER (WHERE {size_condition})
                   AS size_eligible_quote_rows,
               count(*) FILTER (WHERE flag_extreme_spread) AS wide_spread_rows,
               count(*) FILTER (WHERE flag_crossed) AS crossed_quote_rows
        FROM quotes GROUP BY 1 ORDER BY 1
    """).fetchdf()

    boundary_checks = []
    for member in canonical["members"]:
        record = connection.execute("""
            SELECT timestamp_utc, source_row, source_gap_seconds, flag_source_gap
            FROM quotes
            WHERE source_sha256 = ? AND source_row = 1
        """, [member["source_sha256"]]).fetchdf()

        assert len(record) == 1
        assert pd.Timestamp(record.iloc[0]["timestamp_utc"]) == pd.Timestamp(member["utc_min"])
        expected_gap = member["boundary_gap_seconds"]
        observed_gap = record.iloc[0]["source_gap_seconds"]
        if expected_gap is None:
            assert pd.isna(observed_gap)
            assert not record.iloc[0]["flag_source_gap"]
        else:
            np.testing.assert_allclose(
                observed_gap, expected_gap, rtol=0, atol=1e-9
            )
            assert bool(record.iloc[0]["flag_source_gap"]) == (
                expected_gap > parameters["gap_seconds"]
            )
        boundary_checks.append({
            "source_file": member["source_file"],
            "first_timestamp_utc": member["utc_min"],
            "expected_gap_seconds": expected_gap,
            "observed_gap_seconds": (
                None if pd.isna(observed_gap) else float(observed_gap)
            ),
            "flag_source_gap": bool(record.iloc[0]["flag_source_gap"]),
            "validation": "passed",
        })
finally:
    connection.close()

assert int(rows) == sum(audit["rows_scanned"] for audit in audit_states)
assert int(sources) == len(audit_states)
assert int(missing_gaps) == 1
assert pd.Timestamp(first_time) == pd.Timestamp(audit_states[0]["utc_min"])
assert pd.Timestamp(last_time) == pd.Timestamp(audit_states[-1]["utc_max"])
assert canonical_flags == canonical["source_flags"]

year_records = [
    item for item in audit_collection["outputs"]
    if Path(item["path"]).name == "yearly_coverage.csv"
]
if len(year_records) != 1:
    raise RuntimeError("The audit's yearly coverage output is missing.")
audit_year_path = REPO_ROOT / year_records[0]["path"]
if _sha256(audit_year_path) != year_records[0]["sha256"]:
    raise RuntimeError("The audit's yearly coverage table changed.")
audited_years = pd.read_csv(audit_year_path)

compare_columns = [
    "year_utc", "quote_rows", "price_eligible_quote_rows",
    "size_eligible_quote_rows", "wide_spread_rows", "crossed_quote_rows",
]
pd.testing.assert_frame_equal(
    canonical_years[compare_columns].reset_index(drop=True),
    audited_years[compare_columns].reset_index(drop=True),
    check_dtype=False,
)

boundary_table = pd.DataFrame(boundary_checks)
print("YEARLY_CANONICAL_RECONCILIATION")
print(canonical_years.to_string(index=False))
print("\nSOURCE_BOUNDARY_CHECKS")
print(boundary_table.to_string(index=False))
print("\nPASS: counts, quality flags, UTC bounds, and source boundaries reconcile.")

In [ ]:
local_binding_path = create_local_binding(
    REPO_ROOT, history_manifest_path, verify=False
)

months = history_settings["final_test_complete_months"]
if type(months) is not int or months < 1:
    raise RuntimeError("final_test_complete_months must be a positive integer.")

observed_first = pd.Timestamp(canonical["scope_utc_min"]).tz_convert("UTC")
observed_last = pd.Timestamp(canonical["scope_utc_max"]).tz_convert("UTC")
last_complete_month_end = observed_last.normalize().replace(day=1)
final_test_start = last_complete_month_end - pd.DateOffset(months=months)
development_start = observed_first.normalize()

if final_test_start < development_start + pd.DateOffset(years=2):
    raise RuntimeError("Insufficient earlier development history for this reservation.")

full_history_plan = {
    "status": "dates reserved; predictive final evaluation not authorized by execution here",
    "selection_rule": "last configured number of complete UTC calendar months",
    "final_test_complete_months": months,
    "source_collection_id": canonical["job_id"],
    "development": {
        "start_utc": development_start.isoformat(),
        "end_utc_exclusive": final_test_start.isoformat(),
    },
    "reserved_final_test": {
        "start_utc": final_test_start.isoformat(),
        "end_utc_exclusive": last_complete_month_end.isoformat(),
    },
    "trailing_partial_month": {
        "start_utc": last_complete_month_end.isoformat(),
        "observed_last_quote_utc": observed_last.isoformat(),
        "use": "excluded from development; no predictive evaluation",
    },
    "model_choices_frozen_for_final_test": False,
    "final_test_predictively_evaluated": False,
    "h001_status": "pending; development-only testing",
}

plan_path = (
    REPO_ROOT / "artifacts/manifests/history"
    / f"reservation-{canonical['job_id']}.json"
)
if plan_path.exists():
    existing_plan = json.loads(plan_path.read_text(encoding="utf-8"))
    if existing_plan != full_history_plan:
        raise RuntimeError(
            "An existing reservation differs. Review the change before replacing it."
        )
else:
    feature_io.write_json(plan_path, full_history_plan)

print("FULL_HISTORY_DATE_RESERVATION")
print(json.dumps(full_history_plan, indent=2))
print("Local feature-input binding:", local_binding_path.relative_to(REPO_ROOT).as_posix())
print("No final-test predictions or model scores were computed.")

In [ ]:
def relative_path(path):
    return Path(path).resolve().relative_to(REPO_ROOT).as_posix()


def update_section(path, name, content):
    begin = f"<!-- {name}:BEGIN -->"
    finish = f"<!-- {name}:END -->"
    existing = path.read_text(encoding="utf-8") if path.exists() else ""
    if existing.count(begin) != existing.count(finish) or existing.count(begin) > 1:
        raise RuntimeError(f"Malformed generated section: {path.name}")
    block = f"{begin}\n{content.strip()}\n{finish}"
    if begin in existing:
        left, remainder = existing.split(begin, 1)
        _, right = remainder.split(finish, 1)
        text = left + block + right
    else:
        text = existing.rstrip() + "\n\n" + block + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(text)


source_execution_table = pd.DataFrame(source_execution_records)
table_dir = REPO_ROOT / "reports/tables" / canonical["job_id"]
table_dir.mkdir(parents=True, exist_ok=True)

tables = {
    "source_ingestion": source_execution_table,
    "canonical_yearly_reconciliation": canonical_years,
    "cross_source_boundary_checks": boundary_table,
    "partition_index": pd.DataFrame(canonical["parts"]),
}
table_paths = {}
for name, table in tables.items():
    path = table_dir / f"{name}.csv"
    table.to_csv(path, index=False)
    table_paths[name] = relative_path(path)

example_path = REPO_ROOT / "configs/research_config.example.json"
local_path = REPO_ROOT / "configs/research_config.local.json"
example = json.loads(example_path.read_text(encoding="utf-8-sig"))
local = json.loads(local_path.read_text(encoding="utf-8-sig"))

example["history_ingestion"] = history_settings
local["active_history_manifest"] = relative_path(history_manifest_path)
local["history_feature_input_manifest"] = relative_path(local_binding_path)
local["full_history_validation_plan"] = full_history_plan

feature_io.write_json(example_path, example)
feature_io.write_json(local_path, local)

environment_path = REPO_ROOT / "reports/private/validated_history_ingestion_environment.json"
feature_io.write_json(environment_path, {
    "python": platform.python_version(),
    "packages": versions,
    "validation": test_summary,
})

method_note = f"""
### Notebook 08 — full-history canonical ingestion

Converted the audited, nonoverlapping exports to resumable source Parquet
generations using the existing validated ingestion implementation.

The canonical collection references each selected source partition once.
Small immutable overlays correct first-quote gap context at later source
boundaries. Original source partitions and pilot generations are preserved.

No price, size, locked-quote, or wide-spread observations were removed.

Portable collection paths are repository-relative. A separate ignored local
binding supports existing feature-generation APIs.

Executed correctness checks: {test_summary}.
Canonical row counts, quality flags, yearly counts, source boundaries,
partition schemas, and integrity checks passed.

A separate full-history final-test date reservation was recorded using the
last {months} complete UTC calendar months. No predictive final evaluation
was executed. The January pilot plan remains separate.

Full-history features, labels, longer-window development evaluation,
H001 testing, final model freezing, and final interpretation remain pending.
Notebook 08 owner review remains pending.
"""

for relative in ("README.md", "reports/data_quality.md", "reports/research_results.md"):
    update_section(REPO_ROOT / relative, "NOTEBOOK_08", method_note)

update_section(
    REPO_ROOT / "CONTRIBUTIONS.md",
    "NOTEBOOK_08",
    f"""
### Notebook 08

- Raymond Fendi requested the full-history ingestion notebook.
- OpenAI Codex supplied the collection helpers, boundary overlays,
  tests, and documentation.
- Executed correctness checks: {test_summary}.
- All audited records were retained and canonical counts reconciled.
- Full-history test dates were reserved without predictive evaluation.
- Notebook 08 owner review and understanding checkpoint: pending.
- No H001-specific test or full-history forecast model was executed.
""",
)

private_report_path = (
    REPO_ROOT / "reports/private" / f"history-ingestion-{canonical['job_id']}.md"
)
private_report_path.parent.mkdir(parents=True, exist_ok=True)
private_report_path.write_text(
    "# Full-history canonical ingestion\n\n"
    "No predictive models fitted or final test evaluated.\n\n"
    "## Source ingestion\n\n```\n"
    + source_execution_table.to_string(index=False)
    + "\n```\n\n## Source boundaries\n\n```\n"
    + boundary_table.to_string(index=False)
    + "\n```\n\n## Date reservation\n\n```json\n"
    + json.dumps(full_history_plan, indent=2)
    + "\n```\n",
    encoding="utf-8",
)

print("Saved full-history collection selections and execution evidence.")
print("January pilot selections remain unchanged.")

In [ ]:
def json_records(frame):
    return json.loads(frame.to_json(orient="records", date_format="iso"))


native_bytes = sum(
    sum(part["bytes"] for part in state["parts"])
    for state in source_states
)
canonical_bytes = sum(part["bytes"] for part in canonical["parts"])
overlay_bytes = sum(
    (REPO_ROOT / patch["overlay_partition"]).stat().st_size
    for patch in canonical["boundary_patches"]
)

summary = {
    "notebook": NOTEBOOK,
    "collection_id": canonical["job_id"],
    "python_version": platform.python_version(),
    "package_versions": versions,
    "configuration": {
        "source_audit_collection_id": audit_collection["collection_id"],
        "source_count": len(audit_states),
        "source_parameters": parameters,
        "history_settings": history_settings,
        "deduplication": "none; audited timestamps strictly increase and ranges are disjoint",
        "boundary_policy": "later source first quote linked to preceding source endpoint",
    },
    "inputs": {
        "audited_source_rows": sum(state["rows_scanned"] for state in audit_states),
        "raw_source_bytes": sum(
            state["identity"]["source_bytes"] for state in audit_states
        ),
    },
    "outputs": {
        "canonical_quote_rows": int(rows),
        "source_count": int(sources),
        "first_quote_utc": pd.Timestamp(first_time).isoformat(),
        "last_quote_utc": pd.Timestamp(last_time).isoformat(),
        "partition_count": len(canonical["parts"]),
        "boundary_overlay_count": len(canonical["boundary_patches"]),
        "quotes_without_preceding_gap": int(missing_gaps),
        "canonical_quality_flags": canonical_flags,
        "yearly_reconciliation": json_records(canonical_years),
        "boundary_checks": boundary_checks,
    },
    "final_test_reservation": full_history_plan,
    "validation": {
        "imports_from_notebooks_directory": "passed",
        "pytest": test_summary,
        "synthetic_split_vs_combined_consistency": "covered by tests",
        "synthetic_pipeline": "passed",
        "source_hashes_match_audit": "passed",
        "native_ingestion_matches_audit": "passed",
        "canonical_partition_integrity_and_schema": "passed",
        "canonical_row_conservation": "passed",
        "yearly_audit_reconciliation": "passed",
        "cross_source_gap_corrections": "passed",
        "original_source_partitions_preserved": "checked during overlay construction",
        "predictive_final_test_evaluated": False,
        "full_history_models_fitted": False,
        "owner_review": "pending",
    },
    "measurements": {
        "ingestion_and_collection_seconds_this_execution": round(ingestion_runtime, 3),
        "sources_ingested_this_execution": sum(
            not state["cache_hit"] for state in source_states
        ),
        "sources_reused_after_verification": sum(
            state["cache_hit"] for state in source_states
        ),
        "collection_cache_hit": history_collection["cache_hit"],
        "native_source_parquet_bytes": native_bytes,
        "canonical_referenced_parquet_bytes": canonical_bytes,
        "additional_boundary_overlay_bytes": overlay_bytes,
        "maximum_observed_ingestion_batch_bytes": max(
            state["maximum_observed_batch_bytes"] for state in source_states
        ),
        "notebook_elapsed_seconds": round(time.perf_counter() - notebook_started, 3),
    },
    "warnings": [
        "Canonical ingestion does not establish predictive value.",
        "Gap flags include market closures and are not automatic exclusions.",
        "Canonical gap counts include cross-file boundary events omitted from within-file audit counts.",
        "Locked quotes, zero sizes, and wide spreads remain present.",
        "Displayed-size denomination and true side-specific quote ages remain unresolved.",
        "Storage and batch measurements are not total process peak memory.",
        "Final-test dates are reserved; final model choices are not yet frozen.",
        "Full-history feature and label generation remain pending.",
        "Longer-window model benchmarks and H001 testing remain pending.",
        "No achievable-profit conclusion has been established.",
    ],
    "saved_outputs": {
        "active_history_manifest": relative_path(history_manifest_path),
        "local_feature_input_binding": relative_path(local_binding_path),
        "final_test_reservation": relative_path(plan_path),
        "tables": table_paths,
        "private_report": relative_path(private_report_path),
        "validated_environment": relative_path(environment_path),
        "execution_summary": "reports/private/notebook08_execution.json",
    },
}

feature_io.write_json(
    REPO_ROOT / "reports/private/notebook08_execution.json", summary
)
print("NOTEBOOK_08_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))